### Imports
---

In [1]:
import sys
import gc
from utils import data_download, build_aggregates
import numpy as np
import pandas as pd
from pathlib import Path

### Data download
---

In [2]:
data_download.main()

Dataset already in data/raw/ml-32m


### Utility functions
---

In [3]:
log = []

def log_step(
    table: str, 
    step: str, 
    before: int, 
    after: int, 
    note: str=""
) -> None:
    """record to note how many rows are lost in each step

    Args:
        table (str): name of the table
        step (str): step used to remove the rows
        before (int): number of rows before the step
        after (int): number of rows after the step
        note (str, optional): additional notes. Defaults to "".
        
    Returns:
        None
    """
    log.append({
        "table": table,
        "step": step,
        "before": before,
        "after": after,
        "removed": before - after,
        "note": note
    })

In [4]:
BASE = Path("data/raw/ml-32m")

CLEAN = Path("data/clean/ml-32m")
CLEAN.mkdir(parents=True, exist_ok=True) 

files = sorted(BASE.glob("*.csv"))

if not files:
    raise FileNotFoundError(f"No CSV found in {BASE.resolve()}")

for p in files:
    print(f"{p.name:<58} {p.stat().st_size / 1e6:8.1f} MB")
    
#First rows
pd.read_csv(files[0], nrows=5)

def header(path):
    cols = pd.read_csv(
        path, 
        nrows=0, 
        skipinitialspace=True
    ).columns
    
    return tuple(cols.str.strip())

headers = {p.name: header(p) for p in files}

# 1 = every file matches
print("Distinct headers:", len(set(headers.values())))

links.csv                                                       2.0 MB
movies.csv                                                      4.2 MB
ratings.csv                                                   877.1 MB
tags.csv                                                       72.4 MB
Distinct headers: 4


### Expected columns by file
---

In [5]:
EXPECTED = {
    "ratings": ["userId", "movieId", "rating", "timestamp"],
    "tags": ["userId", "movieId", "tag", "timestamp"],
    "movies": ["movieId", "title", "genres"],
    "links": ["movieId", "imdbId", "tmdbId"] 
}

for name, cols in EXPECTED.items():
    real = list(
        pd
        .read_csv(BASE / f"{name}.csv", nrows=0)
        .columns
    )
    
    print(name, "OK" if real == cols else f"Different: {real}")

ratings OK
tags OK
movies OK
links OK


### Loading the datasets
---
Why `rating` goes from `float64` to `float32`: Pandas stores decimals as 
64-bit floats by default, but rating only takes ten values (0.5 to 5.0 in 
steps of 0.5), and float32 represents all of them exactly. Halving the size 
from 8 to 4 bytes per value saves about 128 MB on the 32 million rows of 
the dataset, with no loss of information.

In [6]:
movies = pd.read_csv(
    BASE / "movies.csv",
    skipinitialspace=True
)

links = pd.read_csv(
    BASE / "links.csv",
    skipinitialspace=True
)

tags = pd.read_csv(
    BASE / "tags.csv",
    skipinitialspace=True
)

ratings = pd.read_csv(
    BASE / "ratings.csv",
    skipinitialspace=True,
    
    dtype = {
        "userId": "int32",
        "movieId": "int32",
        "rating": "float32",
        "timestamp": "int32"
    }
)

### Normalization
---

In [7]:
def tables() -> dict[str, pd.DataFrame]:
    return {
        "ratings": ratings, 
        "tags": tags, 
        "movies": movies, 
        "links": links
    }

rename = {
    "userId": "user_id",
    "movieId": "movie_id",
    "imdbId": "imdb_id",
    "tmdbId": "tmdb_id"
}

def norm(col: str) -> str:
    return " ".join(col.strip().lower().split())

def convert_to_timestamp(df: pd.DataFrame) -> None:
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(
            df["timestamp"], 
            unit="s", 
            errors="coerce"
        )

for name, t in tables().items():
    t.columns = [rename.get(c, norm(c)) for c in t.columns]
    convert_to_timestamp(t)
    
    print(f"\n{name}: {t.shape}")
    t.info(memory_usage="deep")
    display(t.head())


ratings: (32000204, 4)
<class 'pandas.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 4 columns):
 #   Column     Dtype         
---  ------     -----         
 0   user_id    int32         
 1   movie_id   int32         
 2   rating     float32       
 3   timestamp  datetime64[us]
dtypes: datetime64[us](1), float32(1), int32(2)
memory usage: 610.4 MB


,user_id,movie_id,rating,timestamp
0,1,17,4.0,1999-12-03 19:24:37
1,1,25,1.0,1999-12-03 19:43:48
2,1,29,2.0,1999-11-22 00:36:16
3,1,30,5.0,1999-12-03 19:24:37
4,1,32,5.0,1999-11-22 00:00:58



tags: (2000072, 4)
<class 'pandas.DataFrame'>
RangeIndex: 2000072 entries, 0 to 2000071
Data columns (total 4 columns):
 #   Column     Dtype         
---  ------     -----         
 0   user_id    int64         
 1   movie_id   int64         
 2   tag        str           
 3   timestamp  datetime64[us]
dtypes: datetime64[us](1), int64(2), str(1)
memory usage: 82.3 MB


,user_id,movie_id,tag,timestamp
0,22,26479,Kevin Kline,2020-03-01 05:01:26
1,22,79592,misogyny,2020-02-12 02:58:17
2,22,247150,acrophobia,2021-05-31 17:51:09
3,34,2174,music,2009-08-09 08:54:24
4,34,2174,weird,2009-08-09 08:55:02



movies: (87585, 3)
<class 'pandas.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   movie_id  87585 non-null  int64
 1   title     87585 non-null  str  
 2   genres    87585 non-null  str  
dtypes: int64(1), str(2)
memory usage: 5.2 MB


,movie_id,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



links: (87585, 3)
<class 'pandas.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   movie_id  87585 non-null  int64  
 1   imdb_id   87585 non-null  int64  
 2   tmdb_id   87461 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 2.0 MB


,movie_id,imdb_id,tmdb_id
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [8]:
for name, t in tables().items():
    print("\n", name)
    print(t.shape)
    display(t.describe(include="all").T)
    print(t.nunique().sort_values())


 ratings
(32000204, 4)


,count,mean,min,25%,50%,75%,max,std
user_id,32000204.0,100278.506411,1.0,50053.0,100297.0,150451.0,200948.0,57949.046233
movie_id,32000204.0,29318.610122,1.0,1233.0,3452.0,44199.0,292757.0,50958.16088
rating,32000204.0,3.540395,0.5,3.0,3.5,4.0,5.0,1.058986
timestamp,32000204,2010-05-30 17:39:59.573262,1995-01-09 11:46:44,2003-04-22 11:53:50,2010-04-30 10:03:32.500000,2017-08-19 15:59:05.250000,2023-10-13 02:29:07,NaN


rating             10
movie_id        84432
user_id        200948
timestamp    25976523
dtype: int64

 tags
(2000072, 4)


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
user_id,2000072.0,NaN,NaN,NaN,81928.586291,22.0,68413.0,78213.0,103698.0,162279.0,38106.498431
movie_id,2000072.0,NaN,NaN,NaN,71893.261729,1.0,4011.0,52328.0,122294.0,292629.0,74803.79499
tag,2000055,140952,sci-fi,10996,NaN,NaN,NaN,NaN,NaN,NaN,NaN
timestamp,2000072,NaN,NaN,NaN,2018-06-13 18:18:05.942505,2005-12-24 13:00:10,2016-09-11 17:25:04.250000,2019-11-18 09:57:42,2021-03-03 02:46:59,2023-10-12 23:56:23,NaN


user_id        15848
movie_id       51323
tag           140952
timestamp    1291250
dtype: int64

 movies
(87585, 3)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
movie_id,87585.0,NaN,NaN,NaN,157651.365519,79013.402099,1.0,112657.0,165741.0,213203.0,292757.0
title,87585,87382,Alone (2020),4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
genres,87585,1798,Drama,12443,NaN,NaN,NaN,NaN,NaN,NaN,NaN


genres       1798
title       87382
movie_id    87585
dtype: int64

 links
(87585, 3)


,count,mean,std,min,25%,50%,75%,max
movie_id,87585.0,1.576514e+05,7.901340e+04,1.0,112657.0,165741.0,213203.0,292757.0
imdb_id,87585.0,2.792840e+06,4.278866e+06,1.0,94642.0,492996.0,3877296.0,29081098.0
tmdb_id,87461.0,2.413823e+05,2.471467e+05,2.0,46836.0,139272.0,381693.0,1186337.0


tmdb_id     87425
movie_id    87585
imdb_id     87585
dtype: int64


### Normalize text values
---

In [9]:
for name, t in tables().items():
    print(name, list(t.select_dtypes(["object", "string"]).columns))

ratings []
tags ['tag']
movies ['title', 'genres']
links []


In [10]:
raw = pd.read_csv(
    BASE / "tags.csv", 
    usecols=["tag"], 
    dtype="string", 
    keep_default_na=False
)

print("empty or spaces:", (raw["tag"].str.strip() == "").sum())

print(
    raw.loc[
        raw["tag"]
        .str.strip()
        .str.lower()
        .isin(["na", "n/a", "null", "nan", "none"]), "tag"
    ].value_counts()
)

empty or spaces: 6
tag
NA      16
null     1
Na       1
Name: count, dtype: int64[pyarrow]


In [11]:
empty_values = {"", "na", "n/a", "null", "nan", "none"}

def clean_text(
    serie: pd.Series, 
    lower: bool = False, 
    placeholders: bool = False
) -> pd.Series:
    """Cleans text values in a serie, representing a DataFrame's column.

    Args:
        serie (pd.Series): serie with text values
        lower (bool, optional): _description_. Defaults to False.
        placeholders (bool, optional): _description_. Defaults to False.

    Returns:
        pd.Series: _description_
    """
    serie = (
        serie
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    
    if lower:
        serie = serie.str.lower()
    
    return (
        serie.mask(serie.isin(empty_values)) 
        if placeholders 
        else serie.mask(serie == "")
    )

text_rules = {
    "tags":   {
        "tag": {"lower": True, "placeholders": True}
    },
    
    "movies": {
        "title": {"lower": False, "placeholders": False},
        "genres": {"lower": False, "placeholders": False}
    }
}

for name, cols in text_rules.items():
    t = tables()[name]
    
    for col, opts in cols.items():
        before_unique = t[col].nunique()
        before_na = t[col].isna().sum()
        
        t[col] = clean_text(t[col], **opts)
        
        print(
            f"{name}.{col}:",
            f"\nuniques {before_unique:,} -> {t[col].nunique():,}",
            f"\nNA {before_na} -> {t[col].isna().sum()}\n",
            "-" * 50
        )

tags.tag: 
uniques 140,952 -> 131,618 
NA 17 -> 24
 --------------------------------------------------
movies.title: 
uniques 87,382 -> 87,382 
NA 0 -> 0
 --------------------------------------------------
movies.genres: 
uniques 1,798 -> 1,798 
NA 0 -> 0
 --------------------------------------------------


### Missing values
---

In [12]:
for name, t in tables().items():
    missing = t.isna().sum()
    missing = missing[missing > 0]
    
    if missing.empty:
        continue
    
    print(f"{name}\n")
    
    print(pd.DataFrame({
        "missing": missing,
        "%": (missing / len(t) * 100).round(3)
    }))
    
    print("-" * 50)

tags

     missing      %
tag       24  0.001
--------------------------------------------------
links

         missing      %
tmdb_id      124  0.142
--------------------------------------------------


In [13]:
before = len(tags)
tags = tags.dropna(subset=["tag"])

log_step(
    table = "tags", 
    step = "dropna", 
    before = before, 
    after = len(tags)
)

before = len(links)
links = links.dropna(subset=["tmdb_id"])

log_step(
    table = "links", 
    step = "dropna", 
    before = before, 
    after = len(links)
)

### Checking impossible values
---
Negative values:

In [14]:
for name, t in tables().items():
    numeric = t.select_dtypes("number")
    negatives = (numeric < 0).sum()
    print(f"{name}: {negatives[negatives > 0]}")

ratings: Series([], dtype: int64)
tags: Series([], dtype: int64)
movies: Series([], dtype: int64)
links: Series([], dtype: int64)


Valid ranges by domain:

In [15]:
# timestamp
timestamp_cols = {
    "ratings": "timestamp",
    "tags": "timestamp"
}

low_date = pd.Timestamp("1995-01-01")
high_date = pd.Timestamp("2023-10-14")

for t, col in timestamp_cols.items():
    s = tables()[t][col]
    out = s.notna() & ~s.between(low_date, high_date) 
    
    print(f"{t}.{col}")
    print(f"NaT: {s.isna().sum()}")
    print(f"min: {s.min()} | max: {s.max()}")
    print(f"out of [{low_date.date()}, {high_date.date()}]: {out.sum()}")
    print("-" * 50)

ratings.timestamp
NaT: 0
min: 1995-01-09 11:46:44 | max: 2023-10-13 02:29:07
out of [1995-01-01, 2023-10-14]: 0
--------------------------------------------------
tags.timestamp
NaT: 0
min: 2005-12-24 13:00:10 | max: 2023-10-12 23:56:23
out of [1995-01-01, 2023-10-14]: 0
--------------------------------------------------


In [16]:
# rating
rating_range = {"rating": (0.5, 5.0)}

for col, (low, high) in rating_range.items():
    bad = ratings[col].notna() & ~ratings[col].between(low, high)
    print(col, bad.sum(), "out of range")

rating 0 out of range


### Duplicated values
---

In [17]:
# An user can't rate the same movie more than once.

# Same with tags: an user can't apply the same tag in the same movie
# more than once.

print(
    "Duplicated ratings:",
    ratings.duplicated(["user_id", "movie_id"]).sum()
)

print(
    "Duplicated movies:",
    f"{movies.duplicated("movie_id").sum()} |",
    f"by title: {movies.duplicated("title").sum()}"
)

print(
    "Duplicated tags:",
    tags.duplicated(["user_id", "movie_id", "tag"]).sum()
)

for name, t in tables().items():
    print(f"Exact duplicates in {name}: {t.duplicated().sum()}")

Duplicated ratings: 0
Duplicated movies: 0 | by title: 203
Duplicated tags: 23
Exact duplicates in ratings: 0
Exact duplicates in tags: 1
Exact duplicates in movies: 0
Exact duplicates in links: 0


### Checking duplicated movies by title
---
They have different `imdb_id` and `tmdb_id`: different movies. They will not
be removed.

In [18]:
dup = movies[
    movies.duplicated("title", keep=False)
].sort_values("title")

dup = dup.merge(
    links, 
    on = "movie_id", 
    how = "left", 
    validate = "one_to_one"
)

display(dup.head(20))

,movie_id,title,genres,imdb_id,tmdb_id
0,230389,2020 (2020),Documentary,13424166.0,765736.0
1,245234,2020 (2020),Comedy,13110216.0,766967.0
2,96456,ATM (2012),Horror|Thriller,1603257.0,89691.0
3,217057,ATM (2012),Comedy|Romance,2191394.0,104651.0
4,136696,Absolution (2015),Action|Adventure|Crime|Thriller,3503840.0,332976.0
5,151797,Absolution (2015),Drama|Thriller,3442634.0,315055.0
6,209761,Adam (2019),Comedy,6316466.0,486233.0
7,223916,Adam (2019),Drama,10199664.0,595875.0
8,588,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,103639.0,812.0
9,114240,Aladdin (1992),Adventure|Animation|Children|Comedy|Fantasy,827990.0,343693.0


Repeated titles in `movies` are different films (distinct `imdb_id` and `tmdb_id`), so they are kept.
36 `tmdb_id` values appear twice in `links`: most look like the same film listed under two `movie_id`, and about 8 look like different films with a wrong `tmdb_id`.
Nothing is removed. `movie_id` is the only key used for joins.

In [19]:
before = len(tags)

tags = tags.drop_duplicates(
    subset = ["user_id", "movie_id", "tag"], 
    keep = "first"
)

log_step(
    table = "tags",
    step = "drop_duplicates",
    before = before,
    after = len(tags),
    note = "key user_id+movie_id+tag (includes the exact duplicate)"
)

print(
    f"'tags' before: {before}",
    f"\n'tags' after: {len(tags)}"
)

'tags' before: 2000048 
'tags' after: 2000025


### Exporting clean datasets
---

In [20]:
for name, t in tables().items():
    filepath = CLEAN / f"{name}.parquet"
    t.to_parquet(filepath, index=False)
    
log_df = pd.DataFrame(log)
display(log_df)
log_df.to_csv(CLEAN / "cleaning_log.csv", index=False)

,table,step,before,after,removed,note
0,tags,dropna,2000072,2000048,24,
1,links,dropna,87585,87461,124,
2,tags,drop_duplicates,2000048,2000025,23,key user_id+movie_id+tag (includes the exact d...


### Building aggregate data for the Data Studio dashboard
---

In [25]:
gc.collect()
AGG = Path("data/agg/ml-32m")
build_aggregates.main(clean=CLEAN, out=AGG)

kpis.csv                    1 rows       0.2 KB
rating_dist.csv            10 rows       0.2 KB
movie_genre.csv        71,354 rows    5464.6 KB
genre_stats.csv            18 rows       0.7 KB
genre_decade.csv          196 rows       6.2 KB
year_stats.csv             29 rows       0.9 KB
hour_weekday.csv          168 rows       2.7 KB
tags_top.csv               20 rows       0.4 KB
long_tail.csv              12 rows       0.6 KB

All consistency checks passed.

n_ratings             32000204
n_users                 200948
n_movies_rated           84432
n_movies_catalog         87585
n_tags                 2000025
n_distinct_tags         131618
global_mean             3.5404
date_min            1995-01-09
date_max            2023-10-13
matrix_density_pct      0.1886

Year flags: [{'year': 1995, 'note': 'very few ratings'}, {'year': 2023, 'note': 'partial year'}]
Top 10 by Bayesian average (genre = All):
                           title  n_ratings  mean_rating  bayes_rating
Shawshank Rede